In [45]:
import numpy as np
import statistics
import pandas as pd
from PIL import Image as PILImage
from pathlib import Path

In [46]:
F_1 = pd.DataFrame(columns=["p", "v", "t"])
V_1 = {}
L = 0
last = 1
ell = 0
folder_dir = r"G:\MorphologyScaleSpace\images"

In [47]:
class Image:
    def __init__(self, data):
        # Store the data as a NumPy array for fast math operations
        self._data = np.array(data)

    @property
    def height(self):
        # The number of rows
        return self._data.shape[0]

    @property
    def width(self):
        # The number of columns
        return self._data.shape[1]

    def __getitem__(self, indices):
        # Allows reading pixels using image[i, j]
        i, j = indices
        return self._data[i, j]

    def __setitem__(self, indices, value):
        # Allows modifying pixels using image[i, j] = value
        i, j = indices
        self._data[i, j] = value

    def as_array(self):
        # Helper to return the raw matrix when you need to do full-image math
        return self._data

In [48]:
def expand(u_t):
    target_h = 2 * u_t.height
    target_w = 2 * u_t.width

    # Initialize the target approximation image (Omega_{t-1})
    u_t_minus_one = Image(np.zeros((target_h, target_w), dtype=int))

    # Boundary Condition: Edge Replication (Zero-order hold)
    def get_p(r, c):
        r_clamp = max(0, min(r, u_t.height - 1))
        c_clamp = max(0, min(c, u_t.width - 1))
        return u_t[r_clamp, c_clamp]

    # Iterate over the TARGET grid (i, j)
    for i in range(target_h):
        for j in range(target_w):

            # Map to SOURCE grid (r, c)
            r = i // 2
            c = j // 2

            # 1. Base Pixel (Direct Copy)
            if i % 2 == 0 and j % 2 == 0:
                u_t_minus_one[i, j] = get_p(r, c)

            # 2. Horizontal Edge Interpolation
            elif i % 2 == 0 and j % 2 == 1:
                median_list = [
                    get_p(r-1, c),   get_p(r-1, c+1),
                    get_p(r, c),     get_p(r, c),     get_p(r, c),
                    get_p(r, c+1),   get_p(r, c+1),   get_p(r, c+1),
                    get_p(r+1, c),   get_p(r+1, c+1)
                ]
                u_t_minus_one[i, j] = statistics.median_low(median_list)

            # 3. Vertical Edge Interpolation
            elif i % 2 == 1 and j % 2 == 0:
                median_list = [
                    get_p(r, c-1),   get_p(r+1, c-1),
                    get_p(r, c),     get_p(r, c),     get_p(r, c),
                    get_p(r+1, c),   get_p(r+1, c),   get_p(r+1, c),
                    get_p(r, c+1),   get_p(r+1, c+1)
                ]
                u_t_minus_one[i, j] = statistics.median_low(median_list)

            # 4. Diagonal Center Interpolation
            else:
                median_list = [
                    get_p(r, c),     get_p(r+1, c),
                    get_p(r, c+1),   get_p(r+1, c+1)
                ]
                u_t_minus_one[i, j] = statistics.median_low(median_list)

    return u_t_minus_one

def decimate(u_t):
    # Base case: The image cannot be decimated further
    if u_t.width == 1 and u_t.height == 1:
        return u_t

    target_h = (u_t.height + 1) // 2
    target_w = (u_t.width + 1) // 2

    # Initialize the target decimated image (Omega_{t+1})
    u_t_plus_one = Image(np.zeros((target_h, target_w), dtype=int))

    # Iterate strictly over the smaller target grid (r, c)
    for r in range(target_h):
        for c in range(target_w):

            # Map back to the source grid (i, j)
            i = 2 * r
            j = 2 * c

            # Direct sub-sampling
            u_t_plus_one[r, c] = u_t[i, j]

    return u_t_plus_one

In [49]:
def compute_residual_t(u_t, est_u_t, t):
    global F_1
    # 1. Dimension Alignment: Crop est_u_t to match u_t strictly
    est_u_t_cropped = est_u_t.as_array()[:u_t.height, :u_t.width]

    # 2. Vectorized Subtraction
    r_t_array = u_t.as_array() - est_u_t_cropped

    # Temporary list to hold the new tuples for this level
    new_tuples = []

    # 3. Iteration and Non-Expansive Filtering
    for i in range(u_t.height):
        for j in range(u_t.width):
            # Skip the sub-sampled base pixels
            if i % 2 != 0 or j % 2 != 0:
                p = (i, j)
                v = r_t_array[i, j]

                new_tuples.append({"p": p, "v": v, "t": t})

    # 4. Batch Update the Feature Set
    if new_tuples:
        new_df = pd.DataFrame(new_tuples)
        # Using concat updates the global DataFrame reference
        F_1 = pd.concat([F_1, new_df], ignore_index=True)

In [50]:
def compute_a_t(t):
    global V_1

    a_t = F_1[F_1['t'] == t]['v'].unique()
    V_1[t] = a_t

In [51]:
def feature_generator(image):
    global F_1, last, L

    done = False
    u_t = image
    t = 1

    while not done:
        u_t_plus_one = decimate(u_t)

        # Base Case condition utilizing dimension check
        if u_t_plus_one.width == u_t.width and u_t_plus_one.height == u_t.height:
            new_tuples = []
            for i in range(u_t.height):
                for j in range(u_t.width):
                    new_tuples.append({"p": (i, j), "v": u_t[i, j], "t": t})

            if new_tuples:
                F_1 = pd.concat([F_1, pd.DataFrame(new_tuples)], ignore_index=True)

            compute_a_t(t)
            last = t
            done = True
            break

        est_u_t = expand(u_t_plus_one)

        compute_residual_t(u_t, est_u_t, t)
        compute_a_t(t)

        t += 1
        u_t = u_t_plus_one

    # L calculation evaluating the Restricted Zeros assumption
    sigma_A_t = 0
    for i in range(1, last + 1):
        sigma_A_t += len(V_1[i])

    zeros_count = len(F_1[F_1["v"] == 0])
    L = (zeros_count - 1) + (sigma_A_t - 1) - 1

In [52]:
def load_and_generate(file_path):
    # 1. Load the native 8-bit grayscale BMP
    raw_img = PILImage.open(file_path)

    # 2. Extract and cast strictly to signed integers
    # This strictly prevents underflow during negative residual calculation
    pixel_matrix = np.array(raw_img, dtype=int)

    # 3. Wrap the matrix in your custom Image class
    u_0 = Image(pixel_matrix)

    # 4. Initialize the theoretical decomposition pipeline
    feature_generator(u_0)

    return u_0.height, u_0.width

In [63]:
def compute_level_dimensions(h_0, w_0, target_t):
    h, w = h_0, w_0
    for _ in range(1, target_t):
        h = (h + 1) // 2
        w = (w + 1) // 2
    return h, w

def reconstruct_residual_t(height_t, width_t, t):
    # 1. Block Memory Allocation: Instantiate Omega_t
    r_t_array = np.zeros((height_t, width_t), dtype=int)

    # 2. Isolate the subset P_t (the surviving features for level t)
    f_t = F_1.loc[F_1['t'] == t]

    # 3. Sparse Matrix Population
    for p, v in zip(f_t["p"], f_t["v"]):
        i, j = p
        r_t_array[i, j] = v

    return Image(r_t_array)

def export_pyramid_gradient(image_name, original_height, original_width):
    global F_1, last

    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)

    # Iterate through every level up to the final base pixel
    for t in range(1, last + 1):
        h, w = compute_level_dimensions(original_height, original_width, t)

        # 1. Allocate an RGBA matrix (4 channels)
        rgba_array = np.zeros((h, w, 4), dtype=np.uint8)

        # 2. Handle the final level (base pixel) vs. residual levels
        if t == last:
            base_df = F_1[F_1["t"] == last]
            if not base_df.empty:
                base_v = int(base_df["v"].values[0])
                base_v_clamped = max(0, min(255, base_v))
                # Base pixel is fully opaque grayscale
                rgba_array[0, 0] = [base_v_clamped, base_v_clamped, base_v_clamped, 255]
        else:
            # Reconstruct the sparse residual matrix
            r_t_image = reconstruct_residual_t(h, w, t)
            r_t_array = r_t_image.as_array()

            # 3. Compute absolute magnitude of the prediction error
            # np.abs safely handles the negative values in the signed integer array
            abs_r_t = np.abs(r_t_array)
            abs_r_t_clamped = np.clip(abs_r_t, 0, 255).astype(np.uint8)

            # Map the absolute magnitude equally to R, G, and B for grayscale
            rgba_array[..., 0] = abs_r_t_clamped
            rgba_array[..., 1] = abs_r_t_clamped
            rgba_array[..., 2] = abs_r_t_clamped

            # Make all calculated residuals strictly opaque by default
            rgba_array[..., 3] = 255

            # Mark strictly decimated coordinates as fully transparent (Alpha = 0)
            rgba_array[0::2, 0::2, 3] = 0

        # 4. Save the isolated image
        output_path = results_dir / f"{image_name}_level_{t}.png"
        PILImage.fromarray(rgba_array, mode="RGBA").save(output_path)

    print(f"Exported {last} absolute magnitude level images for {image_name}.")

In [64]:
def reset_global_state():
    """Strictly resets the global co-domain before a new decomposition."""
    global F_1, V_1, L, last, ell

    # Re-initialize empty state
    F_1 = pd.DataFrame(columns=["p", "v", "t"])
    V_1 = {}
    L = 0
    last = 1
    ell = 0

def process_image_folder(folder_path):
    global L
    # Create a Path object for robust directory handling
    directory = Path(folder_path)

    # Iterate specifically over all BMP files in the folder
    for file_path in directory.glob("*.bmp"):
        print(f"--- Starting Decomposition for: {file_path.name} ---")

        # 1. Purge previous image data
        reset_global_state()


        # 2. Execute pipeline (cast to string for PIL compatibility)
        original_h, original_w = load_and_generate(str(file_path))

        export_pyramid_gradient(file_path.stem, original_h, original_w)

In [66]:
process_image_folder(folder_dir)

--- Starting Decomposition for: lena.bmp ---
Exported 10 absolute magnitude level images for lena.


In [67]:
F_1

,p,v,t
0,"(0, 1)",0,1
1,"(0, 3)",-1,1
2,"(0, 5)",-5,1
3,"(0, 7)",-3,1
4,"(0, 9)",-1,1
...,...,...,...
262139,"(3, 3)",34,8
262140,"(0, 1)",-22,9
262141,"(1, 0)",-59,9
262142,"(1, 1)",-55,9
